# Entrenamiento en Colab

Todo queda en `MyDrive/INTEP/IA/`: el repo, los checkpoints y las gráficas. Solo las imágenes se descomprimen en el disco local de Colab (`/content`), por velocidad.

Requisito: *Entorno de ejecución → Cambiar tipo → T4 GPU*.

## 1. Verificar GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "Sin GPU: Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU -> Guardar, "
    "y vuelve a ejecutar desde esta celda"
)
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive + repo (se clona una sola vez; después solo `git pull`)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

WORK_DIR = "/content/drive/MyDrive/INTEP/IA"
REPO_URL = "https://github.com/JuanMillan7818/eye-disease-detection.git"
PROJECT_DIR = f"{WORK_DIR}/eye-disease-detection"

if os.path.isdir(PROJECT_DIR):
    %cd {PROJECT_DIR}
    !git pull
else:
    %cd {WORK_DIR}
    !git clone {REPO_URL}
    %cd {PROJECT_DIR}
!git log --oneline -1

## 3. Imágenes: zip de Drive → disco local de Colab

In [ ]:
import csv

ZIP_PATH = f"{WORK_DIR}/data_processed.zip"
IMAGE_DIR = "/content/data_processed"  # disco local: mucho más rápido que leer desde Drive

!rm -rf {IMAGE_DIR}
!unzip -q -o "{ZIP_PATH}" -d {IMAGE_DIR}
# Si el zip traía la carpeta adentro (data_processed/data_processed/...), subirla un nivel
!if [ -d {IMAGE_DIR}/data_processed ]; then mv {IMAGE_DIR}/data_processed/* {IMAGE_DIR}/ && rmdir {IMAGE_DIR}/data_processed; fi

# Verificar que existan TODAS las imágenes listadas en splits.csv
rows = list(csv.DictReader(open("splits/splits.csv", encoding="utf-8")))
missing = [r["file"] for r in rows if not os.path.exists(os.path.join(IMAGE_DIR, r["file"]))]
print(f"{len(rows)} imágenes en splits.csv | {len(missing)} faltan")
assert not missing, f"Faltan imágenes, ej. {missing[:5]}: vuelve a crear y subir el zip"

## 4. Entrenar EfficientNet-B0

El checkpoint, el historial y la gráfica se guardan directo en Drive (`models/` y `reports/figures/` del repo).

In [ ]:
!python -m src.train --model b0 --workers 2 --image-dir {IMAGE_DIR}

In [ ]:
from IPython.display import Image
Image("reports/figures/b0_training_curves.png")

## 5. Entrenar EfficientNetV2-S

`--batch-size 16`: con imágenes de 384 px, 32 puede no caber en la T4.

In [ ]:
!python -m src.train --model v2s --workers 2 --batch-size 16 --image-dir {IMAGE_DIR}

In [ ]:
Image("reports/figures/v2s_training_curves.png")

## 6. Experimento: EfficientNetV2-S a 448 px

Se guarda como `v2s_448_*` (no sobrescribe `v2s_best.pth`). Criterio fijado de antemano: se adopta solo si su **val F1 ≥ 0,889**.

In [ ]:
!python -m src.train --model v2s --image-size 448 --workers 2 --batch-size 16 --image-dir {IMAGE_DIR}

In [ ]:
Image("reports/figures/v2s_448_training_curves.png")

## 7. Archivos generados en Drive

In [ ]:
!ls -lh models reports/figures